# Golden set ✨

Mostrando a aplicação do modelo em 5 exemplos

## 0. Configs

### 0.1 Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
import pandas as pd
import numpy as np

from joblib import load

pd.set_option('display.max_columns', None)

### 0.2 Data

In [5]:
df = pd.read_parquet('../data/trusted/tabela_analitica.parquet', engine = 'pyarrow')

df.head()

,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,56,housemaid,married,basic.4y,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,Entre 50 e 60 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,Entre 31 e 40 anos,yes,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,Entre 31 e 40 anos,no,no,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,Entre 50 e 60 anos,no,yes,telephone,05. may,2008,1. mon,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 1. Seleção dos casos

Selecionando amostras de 2010 (teste) tentando maximizar a diversidade de categorias das colunas

In [16]:
colunas = [
    'month',
    'job',
    'education',
    'marital',
    'faixa_etaria',
    'housing',
    'loan',
    'day_of_week',
    'poutcome',
    'contact',
    'y'
]

df_temp = df.drop_duplicates(subset=colunas).copy()

selecionadas = []
valores_usados = {col: set() for col in colunas}

for _ in range(5):

    # Pontuação = quantos valores novos cada linha adicionaria
    scores = df_temp.apply(
        lambda row: sum(
            row[col] not in valores_usados[col]
            for col in colunas
        ),
        axis=1
    )

    # Entre as melhores opções, escolhe uma aleatoriamente
    melhores = scores[scores == scores.max()].index
    idx = melhores.to_series().sample(1, random_state=7).iloc[0]

    linha = df_temp.loc[idx]
    selecionadas.append(idx)

    # Registra as categorias que já apareceram
    for col in colunas:
        valores_usados[col].add(linha[col])

    df_temp = df_temp.drop(index=idx)

golden_set = df.loc[selecionadas]

golden_set

,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
40689,29,admin.,single,university.degree,Entre 26 e 30 anos,no,no,cellular,09. sep,2010,5. fri,1,success,-1.1,94.199,-37.5,0.878,4963.6,no
2305,33,management,married,basic.4y,Entre 31 e 40 anos,yes,yes,telephone,05. may,2008,2. tue,0,nonexistent,1.1,93.994,-36.4,4.856,5191.0,yes
26275,46,self-employed,divorced,basic.9y,Entre 41 e 50 anos,unknown,unknown,cellular,11. nov,2008,4. thu,1,failure,-0.1,93.200,-42.0,4.076,5195.8,no
9487,54,blue-collar,unknown,basic.6y,Entre 50 e 60 anos,no,no,telephone,06. jun,2008,1. mon,0,nonexistent,1.4,94.465,-41.8,4.961,5228.1,no
28003,20,student,single,unknown,Até 25 anos,yes,no,cellular,04. apr,2009,3. wed,0,nonexistent,-1.8,93.075,-47.1,1.498,5099.1,yes


## 2. Predictions

In [19]:
# Carrega todos os artefatos necessários para inferência
bundle = load("../models/lints_bundle.joblib")

preprocessor = bundle["preprocessor"]
lints = bundle["bandit"]
reward_models = bundle["reward_models"]
arms = bundle["arms"]
context_features = bundle["context_features"]

golden_predictions = golden_set.copy()

X_golden = np.asarray(
    preprocessor.transform(golden_predictions[context_features]),
    dtype=np.float64
)

recommended_actions = np.asarray(
    lints.predict(contexts=X_golden)
).reshape(-1)

probabilidades = np.empty(len(golden_predictions))

for arm in arms:
    mask = recommended_actions == arm

    if mask.any():
        probabilidades[mask] = reward_models[arm].predict_proba(X_golden[mask])[:, 1]

golden_predictions["contact_recomendado"] = recommended_actions
golden_predictions["probabilidade_conversao"] = probabilidades

threshold = 0.30

golden_predictions["aceita_oferta"] = np.where(
    golden_predictions["probabilidade_conversao"] >= threshold,
    "yes",
    "no"
)

golden_predictions

,age,job,marital,education,faixa_etaria,housing,loan,contact,month,year,day_of_week,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,contact_recomendado,probabilidade_conversao,aceita_oferta
40689,29,admin.,single,university.degree,Entre 26 e 30 anos,no,no,cellular,09. sep,2010,5. fri,1,success,-1.1,94.199,-37.5,0.878,4963.6,no,cellular,0.895191,yes
2305,33,management,married,basic.4y,Entre 31 e 40 anos,yes,yes,telephone,05. may,2008,2. tue,0,nonexistent,1.1,93.994,-36.4,4.856,5191.0,yes,telephone,0.022030,no
26275,46,self-employed,divorced,basic.9y,Entre 41 e 50 anos,unknown,unknown,cellular,11. nov,2008,4. thu,1,failure,-0.1,93.200,-42.0,4.076,5195.8,no,cellular,0.033209,no
9487,54,blue-collar,unknown,basic.6y,Entre 50 e 60 anos,no,no,telephone,06. jun,2008,1. mon,0,nonexistent,1.4,94.465,-41.8,4.961,5228.1,no,cellular,0.089942,no
28003,20,student,single,unknown,Até 25 anos,yes,no,cellular,04. apr,2009,3. wed,0,nonexistent,-1.8,93.075,-47.1,1.498,5099.1,yes,cellular,0.291072,no
